In [16]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os

VECTORIZER_PATH = "src/vectorizer.pkl"
DB_PATH = "src/fix_database.pkl"
DATA_CSV_PATH = "data/code_bug_fix_pairs.csv"

print(f"Current working directory: {os.getcwd()}")
print(f"Checking paths: \nVectorizer: {os.path.exists(VECTORIZER_PATH)}\nDB: {os.path.exists(DB_PATH)}")

Current working directory: d:\7_semester\SII\Curse\ml_service
Checking paths: 
Vectorizer: True
DB: True


In [17]:
try:
    vectorizer = joblib.load(VECTORIZER_PATH)
    db = joblib.load(DB_PATH)
    
    db_vectors = vectorizer.transform(db["buggy_code"])
    
    print("✅ Модель успешно загружена!")
    print(f"Размер базы знаний: {len(db)} записей")
    print(f"Количество признаков (ngram): {len(vectorizer.get_feature_names_out())}")
    
except Exception as e:
    print(f"❌ Ошибка загрузки: {e}")
    print("Убедитесь, что вы запустили train_model.py хотя бы один раз.")

✅ Модель успешно загружена!
Размер базы знаний: 1021 записей
Количество признаков (ngram): 4648


In [18]:
def check_confidence(query_text: str, top_n=5):
    """
    Анализирует запрос и показывает степень уверенности модели.
    """
    print(f"\n🔎 Анализ запроса: '{query_text}'")
    print("-" * 50)
    
    # 1. Векторизация запроса
    try:
        query_vec = vectorizer.transform([query_text])
    except ValueError:
        print("Ошибка векторизации. Возможно, текст пустой или содержит недопустимые символы.")
        return

    # 2. Расчет сходства (Cosine Similarity)
    # Результат - массив чисел от 0 до 1, где 1 - полное совпадение
    similarities = cosine_similarity(query_vec, db_vectors).flatten()
    
    # 3. Поиск лучших совпадений
    # Получаем индексы сортировки (от меньшего к большему), берем последние top_n и разворачиваем
    best_indices = similarities.argsort()[-top_n:][::-1]
    
    # 4. Вывод результатов
    found_good_match = False
    
    for i, idx in enumerate(best_indices):
        confidence = similarities[idx]
        row = db.iloc[idx]
        
        prefix = "⭐ BEST MATCH" if i == 0 else f"   Option {i+1}"
        
        print(f"{prefix} | Уверенность: {confidence:.4f} ({confidence*100:.1f}%)")
        print(f"   База (buggy_code):  {repr(row['buggy_code'])}")
        print(f"   Исправление:        {repr(row['commit_message'])}")
        print("-" * 30)
        
        if i == 0:
            if confidence >= 0.65:
                print("✅ Вердикт: REPORT SERVICE примет этот ответ.")
            else:
                print("⚠️ Вердикт: Уверенность НИЖЕ порога (0.65). Report Service пойдет в Базу Знаний.")

# Пример использования функции
check_confidence("def foo()")


🔎 Анализ запроса: 'def foo()'
--------------------------------------------------
⭐ BEST MATCH | Уверенность: 1.0000 (100.0%)
   База (buggy_code):  'def foo()'
   Исправление:        'В определении функции пропущено двоеточие.'
------------------------------
✅ Вердикт: REPORT SERVICE примет этот ответ.
   Option 2 | Уверенность: 0.3126 (31.3%)
   База (buggy_code):  "def foo()\n    print('Missing colon in function definition')\n# Sample ID: 4"
   Исправление:        'Added missing parentheses for print function'
------------------------------
   Option 3 | Уверенность: 0.3126 (31.3%)
   База (buggy_code):  "def foo()\n    print('Missing colon in function definition')\n# Sample ID: 4"
   Исправление:        'Added missing parentheses for print function'
------------------------------
   Option 4 | Уверенность: 0.3055 (30.6%)
   База (buggy_code):  "def foo()\n    print('Missing colon in function definition')\n# Sample ID: 10"
   Исправление:        'Indentation fixed for proper code bl

In [19]:
# Тест 1: Синтаксическая ошибка (как в вашем примере)
check_confidence("invalid syntax")

# Тест 2: Ошибка с двоеточием (попробуйте оба варианта, чтобы почувствовать разницу)
check_confidence("def calculate_sum(a, b)") 
check_confidence("def calculate_sum(a, b):")

# Тест 3: Текст ошибки Python (если вы добавили его в CSV по моему совету)
check_confidence("expected ':'")


🔎 Анализ запроса: 'invalid syntax'
--------------------------------------------------
⭐ BEST MATCH | Уверенность: 1.0000 (100.0%)
   База (buggy_code):  'invalid syntax'
   Исправление:        'Проверьте синтаксис: возможно, пропущено двоеточие, скобка или кавычка.'
------------------------------
✅ Вердикт: REPORT SERVICE примет этот ответ.
   Option 2 | Уверенность: 0.0208 (2.1%)
   База (buggy_code):  "my_dict = {'a': 1, 'b': 2}\nfor key value in my_dict:\n    print(key, value)\n# Sample ID: 14"
   Исправление:        'Corrected for-loop unpacking syntax for dictionary'
------------------------------
   Option 3 | Уверенность: 0.0128 (1.3%)
   База (buggy_code):  'unindent does not match any outer indentation level'
   Исправление:        'Ошибка отступа. Смешаны табы и пробелы или неверный уровень вложенности.'
------------------------------
   Option 4 | Уверенность: 0.0000 (0.0%)
   База (buggy_code):  'unexpected indent'
   Исправление:        'Ошибка отступа. Убедитесь, что отс

In [20]:
def analyze_ngrams(text):
    """Показывает, на какие n-граммы разбивается текст"""
    analyzer = vectorizer.build_analyzer()
    tokens = analyzer(text)
    print(f"Текст: '{text}'")
    print(f"Токены (n-граммы): {tokens}")
    
    # Проверим, какие из этих токенов реально есть в словаре модели
    known_tokens = [t for t in tokens if t in vectorizer.vocabulary_]
    print(f"Знакомые модели токены: {known_tokens}")
    print(f"Процент знакомых: {len(known_tokens) / len(tokens) * 100:.1f}%" if tokens else "0%")

# Проверьте на тексте ошибки
analyze_ngrams("invalid syntax")

Текст: 'invalid syntax'
Токены (n-граммы): ['inv', 'nva', 'val', 'ali', 'lid', 'id ', 'd s', ' sy', 'syn', 'ynt', 'nta', 'tax', 'inva', 'nval', 'vali', 'alid', 'lid ', 'id s', 'd sy', ' syn', 'synt', 'ynta', 'ntax', 'inval', 'nvali', 'valid', 'alid ', 'lid s', 'id sy', 'd syn', ' synt', 'synta', 'yntax']
Знакомые модели токены: ['inv', 'nva', 'val', 'ali', 'lid', 'id ', 'd s', ' sy', 'syn', 'ynt', 'nta', 'tax', 'inva', 'nval', 'vali', 'alid', 'lid ', 'id s', 'd sy', ' syn', 'synt', 'ynta', 'ntax', 'inval', 'nvali', 'valid', 'alid ', 'lid s', 'id sy', 'd syn', ' synt', 'synta', 'yntax']
Процент знакомых: 100.0%
